> 在线讲义：[第 59 章 · 神经场景表示：体渲染积分与 α 合成、NeRF 的位置编码、3DGS 的投影与球谐颜色](https://wikieden.github.io/math-and-eval-stats/lessons/59-neural-rendering.html) · 本书《数学与实验统计》

# 第 59 章 · 神经场景表示：体渲染积分与 α 合成、NeRF 的位置编码、3DGS 的投影与球谐颜色

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/59-neural-rendering.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 闭式解 | c(1 − T) + T·c_墙 | `closed_form()` | 0.579272 |
| 第二步 离散 | α 合成 / σδ | `discrete(n)`、`naive_riemann(n)` | 恒等 / 0.874 → 0.583 |
| 第三步 位置编码 | γ(p) | `posenc(0.5)` | 20 维，距离 3.69 |
| 第四步 投影 | fσ/z | `gaussian_2d_std(0.05, z)` | 12.5、5、2.5 像素 |
| 第五步 球谐 | (l + 1)² | `sh_coeffs(3)` | 16 |

例子：一条射线穿过一块密度均匀的“雾”（σ = 2，厚 0.5，颜色 0.8），后面是颜色 0.2 的不透明墙：体渲染积分的闭式解和离散求和；
α 合成的等价写法；NeRF 的位置编码维数和“近的点编码后不再近”；3DGS 里一个各向同性高斯投影到图像的大小；球谐系数个数。
里程碑 ME-M35 的“通用的单射线离散体渲染”“任意 3D 高斯协方差的投影”留给自己做。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np
SIGMA, A, B, C_FOG, C_WALL = 2.0, 1.0, 1.5, 0.8, 0.2

**函数 `closed_form`**：C = ∫ T(t) σ c dt：雾这段贡献 c_fog·(1 − e^{−σ(b−a)})，穿过雾的透射率 e^{−σ(b−a)} 乘墙的颜色。

In [ ]:
def closed_form() -> float:
    """C = ∫ T(t) σ c dt：雾这段贡献 c_fog·(1 − e^{−σ(b−a)})，穿过雾的透射率 e^{−σ(b−a)} 乘墙的颜色。"""
    t = np.exp(-SIGMA * (B - A))
    return C_FOG * (1 - t) + t * C_WALL

**函数 `discrete`**：把雾切成 n 段，每段 α_i = 1 − e^{−σδ}，T_i = Π_{j<i}(1 − α_j)，C = Σ T_i α_i c_i + T_end·c_wall。

In [ ]:
def discrete(n: int) -> float:
    """把雾切成 n 段，每段 α_i = 1 − e^{−σδ}，T_i = Π_{j<i}(1 − α_j)，C = Σ T_i α_i c_i + T_end·c_wall。"""
    delta = (B - A) / n
    alpha = 1 - np.exp(-SIGMA * delta)
    t, c = 1.0, 0.0
    for _ in range(n):
        c += t * alpha * C_FOG
        t *= 1 - alpha
    return c + t * C_WALL

**函数 `naive_riemann`**：如果直接用 σδ 当权重（不用 1 − e^{−σδ}），段数少时会算错。

In [ ]:
def naive_riemann(n: int) -> float:
    """如果直接用 σδ 当权重（不用 1 − e^{−σδ}），段数少时会算错。"""
    delta = (B - A) / n
    t, c = 1.0, 0.0
    for _ in range(n):
        c += t * SIGMA * delta * C_FOG
        t *= np.exp(-SIGMA * delta)
    return c + t * C_WALL

**函数 `posenc`**：NeRF 的位置编码 γ(p) = (sin 2⁰πp, cos 2⁰πp, …, sin 2^{L−1}πp, cos 2^{L−1}πp)。

In [ ]:
def posenc(p: float, levels: int = 10) -> np.ndarray:
    """NeRF 的位置编码 γ(p) = (sin 2⁰πp, cos 2⁰πp, …, sin 2^{L−1}πp, cos 2^{L−1}πp)。"""
    f = 2.0 ** np.arange(levels) * np.pi
    return np.r_[np.sin(f * p), np.cos(f * p)]

**函数 `gaussian_2d_std`**：光轴上深度 z 处、各向同性（标准差 σ）的 3D 高斯：透视投影的雅可比在光轴上是 diag(f/z, f/z)，所以 2D 标准差 = fσ/z 像素。

In [ ]:
def gaussian_2d_std(sigma3d: float, z: float, f: float = 500.0) -> float:
    """光轴上深度 z 处、各向同性（标准差 σ）的 3D 高斯：透视投影的雅可比在光轴上是 diag(f/z, f/z)，所以 2D 标准差 = fσ/z 像素。"""
    return f * sigma3d / z

**函数 `sh_coeffs`**：球谐到 l 阶共 (l + 1)² 个基函数；每个颜色通道一组。

In [ ]:
def sh_coeffs(degree: int) -> int:
    """球谐到 l 阶共 (l + 1)² 个基函数；每个颜色通道一组。"""
    return (degree + 1) ** 2

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 体渲染积分：透射率 × 密度 × 颜色")
print(f"  雾 σ = 2、厚 0.5、颜色 0.8，后面墙颜色 0.2：闭式解 C = {closed_form():.6f}；穿过雾的透射率 {np.exp(-SIGMA * (B - A)):.4f}")

In [ ]:
print("\n第二步 离散化：α = 1 − e^{−σδ} 才对")
for n in (1, 4, 64):
    print(f"  切 {n} 段：α 合成 {discrete(n):.6f}；直接用 σδ 当权重 {naive_riemann(n):.6f}")

In [ ]:
print("\n第三步 位置编码：让网络能表示高频细节")
a, b = posenc(0.50), posenc(0.51)
print(f"  L = 10：编码后 {len(a)} 维（三个坐标共 {3 * len(a)} 维）")
print(f"  0.50 和 0.51 相差 0.01；编码后相差 {np.linalg.norm(a - b):.3f}（最高频那对每 1/512 就转一圈）")

In [ ]:
print("\n第四步 3D 高斯投影到图像")
for z in (2.0, 5.0, 10.0):
    print(f"  标准差 0.05 m 的高斯在 {z} m 处：图像上标准差 {gaussian_2d_std(0.05, z):.1f} 像素")

In [ ]:
print("\n第五步 球谐颜色")
for l in (0, 1, 3):
    print(f"  {l} 阶：{sh_coeffs(l)} 个系数 / 通道，RGB 共 {3 * sh_coeffs(l)} 个")

## 3. 代码里才看得到的

1. **例子的密度是分段常数**：所以 α 合成和闭式解完全相等；密度连续变化时离散化才有误差，要靠更细的采样或分层采样。
2. **高斯只放在光轴上、各向同性**：此时 J 退化成 diag(f/z, f/z)；任意 3D 协方差投影到 2D 的完整计算留给自己做（里程碑 ME-M35）。
3. **没有写一般的单射线渲染函数**：给定任意的 (σ_i, c_i, δ_i) 序列做离散体渲染，也留给里程碑 ME-M35。

## 4. 自己改一改

1. 把雾分成两层（前半 σ = 4、后半 σ = 0），用 `discrete` 的思路重算颜色。
2. 把位置编码的 L 从 10 改成 4，0.50 和 0.51 编码后的距离变成多少？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 透过雾看墙，透射率乘密度乘颜色沿射线积；
> 离散用 1 减 e 的负 σδ，段内常密度就精确；
> 位置编码一串正弦，近的点被拉开，网络才学得出细节；
> 3DGS 高斯投成椭圆，JWΣWᵀJᵀ，再 α 合成；
> 球谐管视角颜色，三阶四十八个系数。